In [1]:
import pandas as pd
import os
import rasterio
import tqdm
import numpy as np
import re
os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity\\CB')

# Generate dummy IDs

In [4]:
file_path = "original.xlsx"

sheets = pd.read_excel(file_path, sheet_name=None)
biota = sheets.get("Biota")

def infer_format(series):
    non_null = series.dropna().astype(str)
    if non_null.empty:
        return ("ID", 4)
    sample = non_null.iloc[0]
    m = re.match(r"^([A-Za-z]*)(\d+)$", sample)
    if m:
        prefix = m.group(1)
        width = len(m.group(2))
    else:
        prefix = ""
        width = max(len(re.findall(r"\d+", s)[-1]) if re.findall(r"\d+", s) else 4 for s in non_null)
    return (prefix, width)

def next_available(existing_nums, start=1):
    n = start
    while n in existing_nums:
        n += 1
    return n

def fill_ids(df, id_col, base_col):
    if id_col not in df.columns or base_col not in df.columns:
        return df

    # Ensure ID column can hold string values without dtype errors
    df[id_col] = df[id_col].astype("string")

    prefix, width = infer_format(df[id_col])
    existing = df[id_col].dropna().astype(str)

    def extract_num(x):
        m = re.match(r"^[A-Za-z]*?(\d+)$", x)
        return int(m.group(1)) if m else None

    used_nums = {n for n in (extract_num(x) for x in existing) if n is not None}

    # map base value -> existing id
    base_to_id = (
        df[[base_col, id_col]]
        .dropna()
        .drop_duplicates(subset=[base_col])
        .set_index(base_col)[id_col]
        .astype(str)
        .to_dict()
    )

    missing_mask = df[id_col].isna()
    for idx, row in df[missing_mask].iterrows():
        base_val = row[base_col]
        if pd.isna(base_val):
            continue
        if base_val in base_to_id:
            df.at[idx, id_col] = base_to_id[base_val]
        else:
            new_num = next_available(used_nums)
            used_nums.add(new_num)
            new_id = f"{prefix}{str(new_num).zfill(width)}"
            df.at[idx, id_col] = new_id
            base_to_id[base_val] = new_id

    # Remove trailing .0 from IDs that were read as floats
    df[id_col] = df[id_col].astype("string").str.replace(r"\.0$", "", regex=True)
    return df

biota = fill_ids(biota, "StationID", "Station")
biota = fill_ids(biota, "DeviceID", "Device")
biota = fill_ids(biota, "SampleID", "Sample")
biota = fill_ids(biota, "SubSampleID", "SubSample")
biota = fill_ids(biota, "SubsetID", "Subset")
biota = fill_ids(biota, "BiotaID", "Biota")

sheets["Biota"] = biota

with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
    for name, df in sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)

In [6]:
# reload workbook
sheets = pd.read_excel(file_path, sheet_name=None)
biota = sheets.get("Biota")

def fill_ids_from_biota(df, biota_df, id_col, base_col):
    if df is None or biota_df is None:
        return df
    if id_col not in df.columns or base_col not in df.columns:
        return df

    mapping = (
        biota_df[[base_col, id_col]]
        .dropna()
        .drop_duplicates(subset=[base_col])
        .set_index(base_col)[id_col]
    )
    df[id_col] = df[id_col].fillna(df[base_col].map(mapping))
    return df

sheet_id_map = {
    "Station": [("StationID", "Station")],
    "Device": [("StationID", "Station"), ("DeviceID", "Device")],
    "Sample": [("StationID", "Station"), ("DeviceID", "Device"), ("SampleID", "Sample")],
    "Subsample": [
        ("StationID", "Station"),
        ("DeviceID", "Device"),
        ("SampleID", "Sample"),
        ("SubSampleID", "SubSample"),
    ],
    "Subset": [
        ("StationID", "Station"),
        ("DeviceID", "Device"),
        ("SampleID", "Sample"),
        ("SubSampleID", "SubSample"),
        ("SubsetID", "Subset"),
    ],
}

for sheet_name, mappings in sheet_id_map.items():
    if sheet_name in sheets:
        df_sheet = sheets[sheet_name]
        for id_col, base_col in mappings:
            df_sheet = fill_ids_from_biota(df_sheet, biota, id_col, base_col)
        sheets[sheet_name] = df_sheet

with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
    for name, df in sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)